In [ ]:
Perform quadrant analysis of friction and vitality and calculate the proportion

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('dalian_85_segments_final.csv')

# Setup style
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10

print("Generating Figure 1...")

# Create figure
fig = plt.figure(figsize=(18, 6))

# --- Figure 1a: Scatterplot ---
ax1 = fig.add_subplot(1, 3, 1)

# Calculate medians for quadrant lines
y3_median = df['Y3'].median()
f_median = df['F'].median()

# Define colors and display labels for each type

type_config = {
    'Commercial': {'color': '#E41A1C', 'label': 'Commercial'},
    'Residential': {'color': '#377EB8', 'label': 'Residential'},
    'Barrier': {'color': '#4DAF4A', 'label': 'Barrier'}
}

# Plot each type
for internal_type, config in type_config.items():
    subset = df[df['Type'] == internal_type]
    ax1.scatter(subset['Y3'], subset['F'],
               s=70, c=config['color'], alpha=0.7,
               edgecolors='white', linewidth=0.8, label=config['label'])

# Add median reference lines
ax1.axvline(y3_median, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax1.axhline(f_median, color='gray', linestyle='--', alpha=0.4, linewidth=1)

# Labels and title
ax1.set_xlabel('Integrated Vitality (Y3)', fontsize=11)
ax1.set_ylabel('Friction Coefficient (F)', fontsize=11)
ax1.set_title('a Vitality-Friction Scatterplot', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 450)
ax1.set_ylim(0, 1.0)
ax1.legend()

# --- Figure 1b: Bar Chart ---
ax2 = fig.add_subplot(1, 3, 2)

quadrants = ['Low-Low', 'Low-High', 'High-Low', 'High-High']
conditions = [
    (df['Y3'] < y3_median) & (df['F'] < f_median),
    (df['Y3'] < y3_median) & (df['F'] >= f_median),
    (df['Y3'] >= y3_median) & (df['F'] < f_median),
    (df['Y3'] >= y3_median) & (df['F'] >= f_median)
]

x = np.arange(len(quadrants))
width = 0.25


for i, (internal_type, config) in enumerate(type_config.items()):
    subset = df[df['Type'] == internal_type]
    counts = [len(subset[cond]) / len(subset) * 100 for cond in conditions]
    bars = ax2.bar(x + (i - 1) * width, counts, width,
                   color=config['color'], alpha=0.8,
                   edgecolor='white', linewidth=0.5, label=config['label'])

ax2.set_xlabel('Quadrant', fontsize=11)
ax2.set_ylabel('Percentage (%)', fontsize=11)
ax2.set_title('b Distribution by Boundary Type', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(quadrants)
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.savefig('figure1_quadrant_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
GAM curve and LOESS verification
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import splrep, splev
from scipy.stats import chi2
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('dalian_85_segments_final.csv')

# Define type mapping: original -> display
TYPE_MAPPING = {
     'Commercial',
     'Residential',
     'Barrier'
}

# Reverse mapping for filtering
TYPE_ORIGINAL = {v: k for k, v in TYPE_MAPPING.items()}


plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# Color scheme for each type
TYPE_COLORS = {
    'Commercial': '#E41A1C',
    'Residential': '#377EB8',
    'Barrier': '#4DAF4A'
}

TYPE_CI_COLORS = {
    'Commercial': '#FEE5D9',
    'Residential': '#DEEBF7',
    'Barrier': '#E5F5E0'
}


def gam_stats_cv(x_data, y_data):
    """GAM statistics with cross-validation for optimal smoothing"""
    n = len(x_data)
    s_values = np.logspace(-2, 2, 15)

    # Cross-validation for optimal s
    kf = KFold(n_splits=min(5, n), shuffle=True, random_state=42)
    cv_errors = []
    for s in s_values:
        errors = []
        for train_idx, val_idx in kf.split(x_data):
            x_train, x_val = x_data[train_idx], x_data[val_idx]
            y_train, y_val = y_data[train_idx], y_data[val_idx]
            try:
                spline = splrep(x_train, y_train, s=s)
                y_pred = splev(x_val, spline)
                errors.append(np.mean((y_val - y_pred) ** 2))
            except:
                errors.append(np.inf)
        cv_errors.append(np.mean(errors))

    best_s = s_values[np.argmin(cv_errors)]

    # Fit with best s
    try:
        spline = splrep(x_data, y_data, s=best_s)
        y_smooth = splev(x_data, spline)
    except:
        from numpy.polynomial import Polynomial
        p = Polynomial.fit(x_data, y_data, 3)
        y_smooth = p(x_data)
        best_s = 1.0

    residuals = y_data - y_smooth
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    # EDF estimation
    if r_squared < 0.05:
        edf = 1.0
    else:
        edf = n * (1 - r_squared) / (1 - (n - 1) / n * (1 - r_squared))
        edf = max(1, min(edf, n - 1))

    if r_squared > 0 and r_squared < 1 and edf > 1 and (n - edf) > 0:
        f_stat = (r_squared / (edf - 1)) / ((1 - r_squared) / (n - edf))
        chi_sq = f_stat * (edf - 1)
        p_value = 1 - chi2.cdf(chi_sq, edf) if chi_sq > 0 else 1.0
    else:
        chi_sq = 0
        p_value = 1.0

    p_value = max(0, min(1, p_value))

    return {
        'edf': edf,
        'chi_sq': chi_sq,
        'p_value': p_value,
        'r_squared': r_squared,
        'best_s': best_s,
        'y_smooth': y_smooth
    }


def loess(x_data, y_data, x_grid, frac=0.5):
    """Simple LOESS: weighted local linear regression"""
    y_smooth = np.zeros_like(x_grid)
    n = len(x_data)
    for i, x0 in enumerate(x_grid):
        # Compute distances
        dist = np.abs(x_data - x0)
        idx_sorted = np.argsort(dist)
        n_neighbors = max(int(frac * n), min(3, n))
        idx = idx_sorted[:n_neighbors]
        x_local = x_data[idx]
        y_local = y_data[idx]
        d_local = dist[idx]
        if d_local.max() > 0:
            weights = (1 - (d_local / d_local.max()) ** 3) ** 3
        else:
            weights = np.ones_like(d_local)

        # Weighted linear regression
        w = weights
        X = np.vstack([np.ones_like(x_local), x_local]).T
        W = np.diag(w)
        try:
            beta = np.linalg.inv(X.T @ W @ X) @ X.T @ W @ y_local
            y_smooth[i] = beta[0] + beta[1] * x0
        except:
            y_smooth[i] = np.mean(y_local)

    return y_smooth


def fit_gam_narrow_ci(x_data, y_data, x_grid, s_factor=1.5):
    try:
        spline = splrep(x_data, y_data, s=s_factor)
        y_smooth = splev(x_grid, spline)
        residuals = y_data - splev(x_data, spline)
        res_std = np.std(residuals)
        ci_lower = y_smooth - 1.96 * res_std
        ci_upper = y_smooth + 1.96 * res_std
        return y_smooth, ci_lower, ci_upper
    except:
        from numpy.polynomial import Polynomial
        p = Polynomial.fit(x_data, y_data, 3)
        y_smooth = p(x_grid)
        residuals = y_data - p(x_data)
        res_std = np.std(residuals)
        return y_smooth, y_smooth - 1.96*res_std, y_smooth + 1.96*res_std


def loess_with_ci(x_data, y_data, x_grid, frac=0.5, n_bootstrap=100):
    """LOESS with bootstrap confidence intervals"""
    y_smooth = loess(x_data, y_data, x_grid, frac)

    # Bootstrap for CI
    n = len(x_data)
    boot_preds = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        x_boot = x_data[idx]
        y_boot = y_data[idx]
        try:
            y_boot_smooth = loess(x_boot, y_boot, x_grid, frac)
            boot_preds.append(y_boot_smooth)
        except:
            continue

    if len(boot_preds) > 10:
        boot_preds = np.array(boot_preds)
        ci_lower = np.percentile(boot_preds, 2.5, axis=0)
        ci_upper = np.percentile(boot_preds, 97.5, axis=0)
    else:
        residuals = y_data - loess(x_data, y_data, x_data, frac)
        res_std = np.std(residuals)
        ci_lower = y_smooth - 1.96 * res_std
        ci_upper = y_smooth + 1.96 * res_std

    return y_smooth, ci_lower, ci_upper


def test_trend_significance(x_data, y_data):
    """Test if trend is significant via linear regression + permutation"""
    from scipy.stats import pearsonr
    corr, p_linear = pearsonr(x_data, y_data)

    # Non-parametric trend test (Mann-Kendall style)
    n = len(x_data)
    s = 0
    for i in range(n-1):
        for j in range(i+1, n):
            if y_data[j] > y_data[i]:
                s += 1
            elif y_data[j] < y_data[i]:
                s -= 1

    # Simple permutation test for trend
    n_perm = 500
    s_perm = []
    y_perm = y_data.copy()
    for _ in range(n_perm):
        np.random.shuffle(y_perm)
        s_perm_i = 0
        for i in range(n-1):
            for j in range(i+1, n):
                if y_perm[j] > y_perm[i]:
                    s_perm_i += 1
                elif y_perm[j] < y_perm[i]:
                    s_perm_i -= 1
        s_perm.append(s_perm_i)

    p_kendall = np.mean(np.abs(s_perm) >= abs(s))

    return {
        'pearson_r': corr,
        'p_linear': p_linear,
        'p_kendall': p_kendall,
        'trend_significant': p_linear < 0.05 or p_kendall < 0.05
    }


def plot_gam_with_stats(ax, x_data, y_data, color, ci_color, xlabel, ylabel, title, stats, show_ci=True):
    x_grid = np.linspace(x_data.min(), x_data.max(), 100)
    y_smooth, ci_l, ci_u = fit_gam_narrow_ci(x_data, y_data, x_grid, s_factor=stats['best_s'])

    ax.plot(x_grid, y_smooth, color=color, linewidth=3, label='GAM fit')
    if show_ci:
        ax.fill_between(x_grid, ci_l, ci_u, color=ci_color, alpha=0.3, label='95% CI')

    p = stats['p_value']
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ax.text(0.05, 0.92, f'EDF = {stats["edf"]:.2f}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.84, f'Chi-sq = {stats["chi_sq"]:.2f}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.76, f'p = {stats["p_value"]:.4f} {sig}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.68, f'R² = {stats["r_squared"]:.3f}', transform=ax.transAxes, fontsize=9)

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.08)
    ax.legend(loc='upper right', fontsize=8)


def plot_loess_with_stats(ax, x_data, y_data, color, ci_color, xlabel, ylabel, title, frac=0.5, show_ci=True):
    x_grid = np.linspace(x_data.min(), x_data.max(), 100)
    y_smooth, ci_l, ci_u = loess_with_ci(x_data, y_data, x_grid, frac=frac, n_bootstrap=100)

    ax.plot(x_grid, y_smooth, color=color, linewidth=3, label='LOESS fit')
    if show_ci:
        ax.fill_between(x_grid, ci_l, ci_u, color=ci_color, alpha=0.3, label='95% CI')

    # Trend test
    trend = test_trend_significance(x_data, y_data)
    sig = '***' if trend['p_linear'] < 0.001 else '**' if trend['p_linear'] < 0.01 else '*' if trend['p_linear'] < 0.05 else 'n.s.'

    ax.text(0.05, 0.92, f'Pearson r = {trend["pearson_r"]:.3f}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.84, f'p (linear) = {trend["p_linear"]:.4f} {sig}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.76, f'p (Kendall) = {trend["p_kendall"]:.4f}', transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.68, f'Trend: {"Significant" if trend["trend_significant"] else "Not significant"}', transform=ax.transAxes, fontsize=9)

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.08)
    ax.legend(loc='upper right', fontsize=8)


PREDICTORS = ['S_Width', 'D_Store', 'Transparency']
RESPONSES = ['F', 'Y3']

PREDICTOR_LABELS = {
    'S_Width': 'Sidewalk width (m)',
    'D_Store': 'Store density (per 100m)',
    'Transparency': 'Transparency'
}

RESPONSE_LABELS = {
    'F': 'Friction coefficient (F)',
    'Y3': 'Integrated vitality (Y3)'
}

print("=" * 80)
print("Generating GAM and LOESS curves for all boundary types...")
print("=" * 80)

all_gam_stats = {}

for type_display, type_original in TYPE_ORIGINAL.items():
    df_type = df[df['Type'] == type_original].copy()
    print(f"\n>>> Processing: {type_display} (n={len(df_type)})")

    type_stats = {}
    for pred in PREDICTORS:
        for resp in RESPONSES:
            x_data = df_type[pred].values
            y_data = df_type[resp].values
            stats = gam_stats_cv(x_data, y_data)
            type_stats[f'{pred}->{resp}'] = stats
    all_gam_stats[type_display] = type_stats


fig_gam, axes_gam = plt.subplots(2, 9, figsize=(24, 8))
fig_gam.suptitle('GAM Analysis by Boundary Type', fontsize=16, fontweight='bold')

for col_idx, type_display in enumerate(['Commercial', 'Residential', 'Barrier']):
    df_type = df[df['Type'] == TYPE_ORIGINAL[type_display]].copy()
    color = TYPE_COLORS[type_display]
    ci_color = TYPE_CI_COLORS[type_display]

    for row_idx, (pred, resp) in enumerate([(p, r) for p in PREDICTORS for r in RESPONSES]):
        ax = axes_gam[row_idx, col_idx * 3 + (row_idx % 3) if row_idx < 2 else col_idx * 3 + 2]

        x_data = df_type[pred].values
        y_data = df_type[resp].values
        stats = all_gam_stats[type_display][f'{pred}->{resp}']

        title = f'{type_display}: {pred} -> {resp}'
        xlabel = PREDICTOR_LABELS[pred]
        ylabel = RESPONSE_LABELS[resp]

        plot_gam_with_stats(ax, x_data, y_data, color, ci_color, xlabel, ylabel, title, stats)

plt.tight_layout(pad=2.0)
plt.savefig('GAM_All_Types.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ GAM_All_Types.png saved")


fig_loess, axes_loess = plt.subplots(2, 9, figsize=(24, 8))
fig_loess.suptitle('LOESS Analysis by Boundary Type', fontsize=16, fontweight='bold')

for col_idx, type_display in enumerate(['Commercial', 'Residential', 'Barrier']):
    df_type = df[df['Type'] == TYPE_ORIGINAL[type_display]].copy()
    color = TYPE_COLORS[type_display]
    ci_color = TYPE_CI_COLORS[type_display]

    for row_idx, (pred, resp) in enumerate([(p, r) for p in PREDICTORS for r in RESPONSES]):
        ax = axes_loess[row_idx, col_idx * 3 + (row_idx % 3) if row_idx < 2 else col_idx * 3 + 2]

        x_data = df_type[pred].values
        y_data = df_type[resp].values

        title = f'{type_display}: {pred} -> {resp}'
        xlabel = PREDICTOR_LABELS[pred]
        ylabel = RESPONSE_LABELS[resp]

        plot_loess_with_stats(ax, x_data, y_data, color, ci_color, xlabel, ylabel, title, frac=0.5)

plt.tight_layout(pad=2.0)
plt.savefig('LOESS_All_Types.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ LOESS_All_Types.png saved")


table_data = []
for type_display in ['Commercial', 'Residential', 'Barrier']:
    for pred in PREDICTORS:
        for resp in RESPONSES:
            stats = all_gam_stats[type_display][f'{pred}->{resp}']
            p = stats['p_value']
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
            shape = 'Nonlinear' if stats['edf'] > 2.0 else 'Weak NL' if stats['edf'] > 1.3 else 'Linear'
            table_data.append({
                'Type': type_display,
                'Predictor': pred,
                'Response': resp,
                'EDF': f'{stats["edf"]:.2f}',
                'Chi-sq': f'{stats["chi_sq"]:.2f}',
                'P-value': f'{stats["p_value"]:.4f}',
                'Sig': sig,
                'R2': f'{stats["r_squared"]:.3f}',
                'Shape': shape
            })

df_table = pd.DataFrame(table_data)
df_table.to_csv('GAM_Trend_Summary_All_Types.csv', index=False)
print("\n✅ Table exported to: GAM_Trend_Summary_All_Types.csv")
print("\n" + df_table.to_string(index=False))


print("\n" + "=" * 80)
print("SUMMARY: Significant Trends by Boundary Type")
print("=" * 80)

for type_display in ['Commercial', 'Residential', 'Barrier']:
    print(f"\n>>> {type_display}:")
    df_sub = df_table[df_table['Type'] == type_display]
    for _, row in df_sub.iterrows():
        print(f"  {row['Predictor']} -> {row['Response']}: {row['Sig']} (EDF={row['EDF']}, R2={row['R2']}, Shape={row['Shape']})")

print("\n" + "=" * 80)
print("Legend: *** p<0.001, ** p<0.01, * p<0.05, n.s. p>=0.05")
print("EDF > 2.0 = Strong Nonlinear, EDF > 1.3 = Weak Nonlinear")
print("=" * 80)

In [ ]:
Significance calculation and robustness test of hysteresis effects on transparency, aspect ratio, and green visual acuity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')


from google.colab import files
print("Please upload dalian_sequence_final.xlsx")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

TYPE_MAPPING = {
     'Commercial',
     'Residential',
     'Barrier'
}

df['Type_Display'] = df['Type'].map(TYPE_MAPPING)

print(f"Data loaded: {len(df)} rows")
print(f"Segments: {df['Segment_ID'].nunique()}")
print(f"Type distribution:\n{df['Type_Display'].value_counts()}")
print(f"\nStatic distribution:\n{df['Static'].value_counts()}")
print(f"Stationary ratio: {df['Static'].sum() / len(df):.2%}")


def sig_star(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

def compute_lag_effects(subset, predictor, lags, n_bootstrap=100):
    coefs = []
    pvals = []
    ses = []

    for lag in lags:
        sub = subset[subset['Time_Lag'] == lag]

        if len(sub) < 10:
            coefs.append(np.nan)
            pvals.append(np.nan)
            ses.append(np.nan)
            continue

        X = sub[[predictor]].values
        y = sub['Static'].values

        if len(np.unique(y)) < 2:
            coefs.append(np.nan)
            pvals.append(np.nan)
            ses.append(np.nan)
            continue

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        model = LogisticRegression(max_iter=1000)
        model.fit(X_scaled, y)
        coef = model.coef_[0][0]

        boot_coefs = []
        for _ in range(n_bootstrap):
            idx = np.random.choice(len(X_scaled), len(X_scaled), replace=True)
            X_boot = X_scaled[idx]
            y_boot = y[idx]
            try:
                m_boot = LogisticRegression(max_iter=1000)
                m_boot.fit(X_boot, y_boot)
                boot_coefs.append(m_boot.coef_[0][0])
            except:
                continue

        if len(boot_coefs) > 10:
            se = np.std(boot_coefs)
            if se > 0:
                z = coef / se
                p = 2 * (1 - norm.cdf(abs(z)))
            else:
                p = 1.0
        else:
            se = np.nan
            p = 1.0

        coefs.append(coef)
        pvals.append(p)
        ses.append(se)

    return np.array(coefs), np.array(pvals), np.array(ses)


def compute_lag_effects_permutation(subset, predictor, lags, n_perm=5000):
    """
    Permutation test for lag effects.
    Returns p_value based on null distribution of shuffled Static labels.
    """
    pvals_perm = []

    for lag in lags:
        sub = subset[subset['Time_Lag'] == lag]

        if len(sub) < 10:
            pvals_perm.append(np.nan)
            continue

        X = sub[[predictor]].values
        y = sub['Static'].values

        if len(np.unique(y)) < 2:
            pvals_perm.append(np.nan)
            continue

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Original coefficient
        model = LogisticRegression(max_iter=1000)
        model.fit(X_scaled, y)
        coef_orig = model.coef_[0][0]

        # Permutation: shuffle y labels
        coef_perm = []
        for _ in range(n_perm):
            y_shuffled = np.random.permutation(y)
            try:
                m_perm = LogisticRegression(max_iter=1000)
                m_perm.fit(X_scaled, y_shuffled)
                coef_perm.append(m_perm.coef_[0][0])
            except:
                continue

        if len(coef_perm) > 10:
            # Two-tailed p-value: proportion of permuted coefficients more extreme than original
            p_perm = np.mean(np.abs(coef_perm) >= abs(coef_orig))
        else:
            p_perm = 1.0

        pvals_perm.append(p_perm)

    return np.array(pvals_perm)


def identify_peak_lags(coef, sig, lags, predictor_name):
    """Identify significant positive and negative peak lags"""
    valid = np.where(~np.isnan(coef))[0]
    results = []

    if len(valid) == 0:
        return results

    # Find significant lags (p < 0.05)
    sig_idx = [i for i in valid if sig[i] and sig[i] not in ['', 'n.s.']]

    if len(sig_idx) == 0:
        return results

    # Positive peaks
    pos_idx = [i for i in sig_idx if coef[i] > 0]
    if pos_idx:
        max_pos_idx = pos_idx[np.argmax(coef[pos_idx])]
        results.append({
            'direction': 'positive',
            'lag': lags[max_pos_idx],
            'coef': coef[max_pos_idx],
            'sig': sig[max_pos_idx]
        })

    # Negative peaks
    neg_idx = [i for i in sig_idx if coef[i] < 0]
    if neg_idx:
        min_neg_idx = neg_idx[np.argmin(coef[neg_idx])]
        results.append({
            'direction': 'negative',
            'lag': lags[min_neg_idx],
            'coef': coef[min_neg_idx],
            'sig': sig[min_neg_idx]
        })

    return results


lags = np.arange(-10, 11)
predictors = ['Delta_Trans', 'Delta_View', 'Delta_HD']
predictor_labels = {'Delta_Trans': 'Transparency', 'Delta_View': 'Green view', 'Delta_HD': 'H/D'}

type_configs = [
    ('Commercial', '#E41A1C'),
    ('Residential', '#377EB8'),
    ('Barrier', '#4DAF4A')
]

all_results = {}
all_perm_results = {}

for type_display, color in type_configs:
    original_type = [k for k, v in TYPE_MAPPING.items() if v == type_display][0]

    print(f"\nComputing {type_display}...")
    df_type = df[df['Type'] == original_type]
    n_seg = df_type['Segment_ID'].nunique()
    print(f"  Rows: {len(df_type)}, Segments: {n_seg}")

    type_results = {}
    type_perm = {}

    for pred in predictors:
        # Original bootstrap
        coef, p, se = compute_lag_effects(df_type, pred, lags)
        type_results[pred] = {
            'coef': coef,
            'p': p,
            'se': se,
            'sig': [sig_star(pv) for pv in p]
        }

        # Permutation test (n=5000)
        p_perm = compute_lag_effects_permutation(df_type, pred, lags, n_perm=5000)
        type_perm[pred] = {
            'p_perm': p_perm,
            'sig_perm': [sig_star(pv) for pv in p_perm]
        }

    all_results[type_display] = type_results
    all_perm_results[type_display] = type_perm



print("\n" + "=" * 80)
print("Exporting coefficient tables with permutation p-values")
print("=" * 80)

with pd.ExcelWriter('Lag_Effect_Coefficients_with_Permutation.xlsx') as writer:
    for type_display, color in type_configs:
        res = all_results[type_display]
        perm = all_perm_results[type_display]

        table = {'Time_Lag': lags}
        for pred in predictors:
            label = {'Delta_Trans': 'Trans', 'Delta_View': 'View', 'Delta_HD': 'HD'}[pred]
            table[f'{label}_coef'] = res[pred]['coef']
            table[f'{label}_p_bootstrap'] = res[pred]['p']
            table[f'{label}_sig_bootstrap'] = res[pred]['sig']
            table[f'{label}_p_permutation'] = perm[pred]['p_perm']
            table[f'{label}_sig_permutation'] = perm[pred]['sig_perm']

        df_table = pd.DataFrame(table)
        print(f"\n[{type_display}] First 5 rows:")
        print(df_table.head(5).to_string(index=False))
        df_table.to_excel(writer, sheet_name=type_display, index=False)

print("\nExported: Lag_Effect_Coefficients_with_Permutation.xlsx")



print("\n" + "=" * 80)
print("Generating Supplementary Table 5: Robustness check")
print("=" * 80)

summary_data = []

for type_display, color in type_configs:
    res = all_results[type_display]
    perm = all_perm_results[type_display]
    original_type = [k for k, v in TYPE_MAPPING.items() if v == type_display][0]
    n_seg = df[df['Type'] == original_type]['Segment_ID'].nunique()

    for pred in predictors:
        coef = res[pred]['coef']
        p_boot = res[pred]['p']
        sig_boot = res[pred]['sig']
        p_perm = perm[pred]['p_perm']
        sig_perm = perm[pred]['sig_perm']

        # Find significant peaks
        peaks = identify_peak_lags(coef, sig_boot, lags, pred)

        for peak in peaks:
            lag = peak['lag']
            # Get permutation p-value at this lag
            lag_idx = np.where(lags == lag)[0][0]
            p_perm_val = p_perm[lag_idx] if not np.isnan(p_perm[lag_idx]) else 1.0

            summary_data.append({
                'Boundary_Type': type_display,
                'Predictor': predictor_labels[pred],
                'Direction': peak['direction'],
                'Peak_Lag': f'+{lag}s' if lag >= 0 else f'{lag}s',
                'Beta': f'{peak["coef"]:.4f}',
                'Sig_Bootstrap': peak['sig'],
                'p_Bootstrap': p_boot[lag_idx],
                'p_Permutation': p_perm_val,
                'Sig_Permutation': sig_perm[lag_idx] if not np.isnan(p_perm_val) else '',
                'Robust': 'Yes' if p_perm_val < 0.05 else 'No'
            })

df_supp_table = pd.DataFrame(summary_data)

# Reorder columns for readability
column_order = ['Boundary_Type', 'Predictor', 'Direction', 'Peak_Lag',
                'Beta', 'Sig_Bootstrap', 'p_Bootstrap',
                'p_Permutation', 'Sig_Permutation', 'Robust']
df_supp_table = df_supp_table[column_order]

print("\nSupplementary Table 5:")
print(df_supp_table.to_string(index=False))

# Export to Excel
with pd.ExcelWriter('Supplementary_Table5_Robustness_Check.xlsx') as writer:
    df_supp_table.to_excel(writer, sheet_name='Table5_Robustness', index=False)

print("\nExported: Supplementary_Table5_Robustness_Check.xlsx")



print("\nGenerating heatmaps...")

plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

var_labels = ['Delta_Trans', 'Delta_View', 'Delta_HD']
var_display = ['Transparency', 'Green view', 'H/D']

for idx, (type_display, color) in enumerate(type_configs):
    ax = axes[idx]
    res = all_results[type_display]

    heatmap = np.array([
        res['Delta_Trans']['coef'],
        res['Delta_View']['coef'],
        res['Delta_HD']['coef']
    ])

    im = ax.imshow(heatmap, cmap='RdBu_r', aspect='auto',
                   vmin=-1.0, vmax=1.0, interpolation='nearest')

    ax.set_xticks(np.arange(0, 21, 2))
    ax.set_xticklabels(np.arange(-10, 11, 2))
    ax.set_yticks(np.arange(3))
    ax.set_yticklabels(var_display, fontsize=9)
    ax.set_xlabel('Time lag (seconds)', fontsize=11)
    ax.set_ylabel('Visual change rate', fontsize=11)

    original_type = [k for k, v in TYPE_MAPPING.items() if v == type_display][0]
    n_seg = df[df['Type'] == original_type]['Segment_ID'].nunique()
    ax.set_title(f'{type_display} (n={n_seg} segments)', fontsize=12, fontweight='bold')

    sig_matrix = np.array([
        res['Delta_Trans']['sig'],
        res['Delta_View']['sig'],
        res['Delta_HD']['sig']
    ])

    for i in range(3):
        for j in range(21):
            sig = sig_matrix[i, j]
            if sig:
                val = heatmap[i, j]
                txt_color = 'white' if abs(val) > 0.4 else 'black'
                ax.text(j, i, sig, ha='center', va='center', color=txt_color,
                        fontsize=9, fontweight='bold')

    ax.axvline(10, color='yellow', linestyle='--', linewidth=1.5, alpha=0.5)
    ax.text(10, -0.6, '0s', ha='center', fontsize=8, color='black')

    cbar = plt.colorbar(im, ax=ax, label='Coefficient (beta)', fraction=0.046, pad=0.04)
    cbar.set_ticks([-1.0, -0.5, 0, 0.5, 1.0])

plt.tight_layout()
plt.savefig('Figure3_Heatmaps.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("Saved: Figure3_Heatmaps.png")



print("\nGenerating response curves...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

curve_colors = {'Delta_Trans': '#E41A1C', 'Delta_View': '#FF7F00', 'Delta_HD': '#984EA3'}
curve_labels = {'Delta_Trans': 'Transparency', 'Delta_View': 'Green view', 'Delta_HD': 'H/D'}
curve_markers = {'Delta_Trans': 'o', 'Delta_View': 's', 'Delta_HD': '^'}

for idx, (type_display, color) in enumerate(type_configs):
    ax = axes[idx]
    res = all_results[type_display]

    for pred in predictors:
        coef = res[pred]['coef']
        sig = res[pred]['sig']

        ax.plot(lags, coef, marker=curve_markers[pred], linestyle='-',
                color=curve_colors[pred], linewidth=2, markersize=6,
                label=curve_labels[pred])

        for j, (lag, val, s) in enumerate(zip(lags, coef, sig)):
            if s:
                y_offset = 0.06 if val > 0 else -0.08
                ax.text(lag, val + y_offset, s, fontsize=8, color=curve_colors[pred],
                        fontweight='bold', ha='center')

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)

    original_type = [k for k, v in TYPE_MAPPING.items() if v == type_display][0]
    n_seg = df[df['Type'] == original_type]['Segment_ID'].nunique()
    ax.set_xlabel('Time lag (seconds)', fontsize=11)
    ax.set_ylabel('Coefficient (beta)', fontsize=11)
    ax.set_title(f'{type_display} (n={n_seg} segments)', fontsize=12, fontweight='bold')
    ax.set_xlim(-10.5, 10.5)
    ax.set_ylim(-1.2, 1.2)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.12)

plt.tight_layout()
plt.savefig('Figure3_Response_Curves.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("Saved: Figure3_Response_Curves.png")



print("\n" + "=" * 80)
print("Key peak statistics with permutation validation")
print("=" * 80)

for type_display, color in type_configs:
    res = all_results[type_display]
    perm = all_perm_results[type_display]
    original_type = [k for k, v in TYPE_MAPPING.items() if v == type_display][0]
    n_seg = df[df['Type'] == original_type]['Segment_ID'].nunique()

    print(f"\n[{type_display}] (n={n_seg})")

    for pred in predictors:
        coef = res[pred]['coef']
        sig = res[pred]['sig']
        p_boot = res[pred]['p']
        p_perm_arr = perm[pred]['p_perm']

        valid = np.where(~np.isnan(coef))[0]

        if len(valid) > 0:
            max_idx = valid[np.argmax(coef[valid])]
            min_idx = valid[np.argmin(coef[valid])]

            pred_label = {'Delta_Trans': 'Transparency', 'Delta_View': 'Green view', 'Delta_HD': 'H/D'}[pred]
            print(f"  {pred_label}:")
            print(f"    Max positive: +{lags[max_idx]}s, beta={coef[max_idx]:.4f} {sig[max_idx]}, p_perm={p_perm_arr[max_idx]:.4f}")
            print(f"    Max negative: +{lags[min_idx]}s, beta={coef[min_idx]:.4f} {sig[min_idx]}, p_perm={p_perm_arr[min_idx]:.4f}")

print("\n" + "=" * 80)
print("Exported files:")
print("  1. Lag_Effect_Coefficients_with_Permutation.xlsx")
print("  2. Supplementary_Table5_Robustness_Check.xlsx")
print("  3. Figure3_Heatmaps.png")
print("  4. Figure3_Response_Curves.png")
print("=" * 80)

In [ ]:
Calculate the inter group differences between meeting and not meeting the standards after grouping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind, shapiro, levene
import warnings
warnings.filterwarnings('ignore')


plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['lines.antialiased'] = True
plt.rcParams['patch.antialiased'] = True

=
df = pd.read_excel('dalian_85_segments_final_with_Y1Y2.xlsx')

print("=" * 60)
print("Data loaded successfully")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")


TYPE_MAPPING = {
    'Building-service': 'Commercial',
    'Building-life': 'Residential',
    'Fenced': 'Barrier'
}

df['Type_Display'] = df['Type'].map(TYPE_MAPPING)

# Define standard conditions using original type names
df['is_life'] = df['Type'] == 'Building-life'       # Residential
df['is_fenced'] = df['Type'] == 'Fenced'            # Barrier
df['is_service'] = df['Type'] == 'Building-service' # Commercial

# Living type (Residential): S_Width between 10.3 and 11.8
df['width_standard'] = df['S_Width'].between(10.3, 11.8)
df['life_standard'] = df['is_life'] & df['width_standard']

# Fenced type (Barrier): S_Length > 150
df['length_standard'] = df['S_Length'] > 150
df['fenced_standard'] = df['is_fenced'] & df['length_standard']

# Combined grouping (using display names)
df['group'] = 'Other'
df.loc[df['life_standard'], 'group'] = 'Residential_Standard'
df.loc[df['is_life'] & ~df['width_standard'], 'group'] = 'Residential_NonStandard'
df.loc[df['fenced_standard'], 'group'] = 'Barrier_Standard'
df.loc[df['is_fenced'] & ~df['length_standard'], 'group'] = 'Barrier_NonStandard'

df['standard_label'] = 'Non-standard'
df.loc[df['life_standard'] | df['fenced_standard'], 'standard_label'] = 'Standard'

print("\n" + "=" * 60)
print("Group Statistics")
print("=" * 60)
print(df['group'].value_counts())
print("\nStandard vs Non-standard:")
print(df['standard_label'].value_counts())


def perform_test(group1, group2, var_name):
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    median1, median2 = np.median(group1), np.median(group2)
    max1, max2 = np.max(group1), np.max(group2)
    min1, min2 = np.min(group1), np.min(group2)

    if n1 < 3 or n2 < 3:
        return {
            'variable': var_name,
            'test_method': 'Insufficient sample',
            'p_value': np.nan,
            'standard_n': n1, 'standard_mean': mean1, 'standard_std': std1,
            'standard_median': median1, 'standard_max': max1, 'standard_min': min1,
            'nonstandard_n': n2, 'nonstandard_mean': mean2, 'nonstandard_std': std2,
            'nonstandard_median': median2, 'nonstandard_max': max2, 'nonstandard_min': min2
        }

    shapiro1 = shapiro(group1)[1]
    shapiro2 = shapiro(group2)[1]
    levene_p = levene(group1, group2)[1]

    if shapiro1 > 0.05 and shapiro2 > 0.05:
        stat, p = ttest_ind(group1, group2, equal_var=levene_p > 0.05)
        test_name = 't-test'
        stat_name = 't'
    else:
        stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
        test_name = 'Mann-Whitney U'
        stat_name = 'U'

    if test_name == 't-test':
        pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
        effect_size = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
        effect_name = "Cohen's d"
    else:
        z_score = (stat - n1*n2/2) / np.sqrt(n1*n2*(n1+n2+1)/12)
        effect_size = abs(z_score) / np.sqrt(n1+n2)
        effect_name = "r"

    return {
        'variable': var_name,
        'test_method': test_name,
        'stat_name': stat_name,
        'statistic': stat,
        'p_value': p,
        'effect_size': effect_size,
        'effect_name': effect_name,
        'standard_n': n1, 'standard_mean': mean1, 'standard_std': std1,
        'standard_median': median1, 'standard_max': max1, 'standard_min': min1,
        'nonstandard_n': n2, 'nonstandard_mean': mean2, 'nonstandard_std': std2,
        'nonstandard_median': median2, 'nonstandard_max': max2, 'nonstandard_min': min2,
        'normality_standard': shapiro1,
        'normality_nonstandard': shapiro2,
        'levene_p': levene_p
    }


variables = ['F', 'Y1_Cross', 'Y2_Stay', 'Y3_Check']
all_results = []

# Use display names for output
for typ_display, typ_original in [('Residential', 'Building-life'), ('Barrier', 'Fenced')]:
    if typ_original == 'Building-life':
        df_sub = df[df['is_life']]
        standard_cond = df_sub['width_standard']
    else:  # Barrier / Fenced
        df_sub = df[df['is_fenced']]
        standard_cond = df_sub['length_standard']

    for var in variables:
        group1 = df_sub[standard_cond][var].values
        group2 = df_sub[~standard_cond][var].values
        result = perform_test(group1, group2, var)
        result['street_type'] = typ_display
        all_results.append(result)


print("\n" + "=" * 80)
print("RESULTS: Standard vs Non-standard Statistical Summary")
print("=" * 80)

summary_data = []
boxplot_data = []

for r in all_results:
    if not np.isnan(r['p_value']):
        p = r['p_value']
        if p < 0.001:
            sig = '***'
        elif p < 0.01:
            sig = '**'
        elif p < 0.05:
            sig = '*'
        else:
            sig = 'n.s.'

        if r['standard_mean'] < r['nonstandard_mean']:
            direction = 'Standard < Non-standard'
        elif r['standard_mean'] > r['nonstandard_mean']:
            direction = 'Standard > Non-standard'
        else:
            direction = 'No difference'

        summary_data.append({
            'Street Type': r['street_type'],
            'Variable': r['variable'],
            'Std_N': r['standard_n'],
            'Std_Mean±SD': f"{r['standard_mean']:.2f}±{r['standard_std']:.2f}",
            'Std_Median': f"{r['standard_median']:.2f}",
            'NonStd_N': r['nonstandard_n'],
            'NonStd_Mean±SD': f"{r['nonstandard_mean']:.2f}±{r['nonstandard_std']:.2f}",
            'NonStd_Median': f"{r['nonstandard_median']:.2f}",
            'Test': r['test_method'],
            'Statistic': f"{r['statistic']:.3f}",
            'p-value': f"{p:.4f}",
            'Sig': sig,
            'Effect Size': f"{r['effect_size']:.3f}",
            'Effect Type': r['effect_name'],
            'Direction': direction
        })

        for group, prefix in [('standard', 'Std'), ('nonstandard', 'NonStd')]:
            boxplot_data.append({
                'Street Type': r['street_type'],
                'Variable': r['variable'],
                'Group': 'Standard' if group == 'standard' else 'Non-standard',
                'N': r[f'{group}_n'],
                'Mean': round(r[f'{group}_mean'], 2),
                'SD': round(r[f'{group}_std'], 2),
                'Median': round(r[f'{group}_median'], 2),
                'Max': round(r[f'{group}_max'], 2),
                'Min': round(r[f'{group}_min'], 2),
                'p-value': round(p, 4),
                'Significance': sig,
                'Effect Size': round(r['effect_size'], 3),
                'Effect Type': r['effect_name']
            })

df_summary = pd.DataFrame(summary_data)
df_boxplot = pd.DataFrame(boxplot_data)

print("\n" + df_summary.to_string(index=False))

==
with pd.ExcelWriter('Statistical_Results_Standard_vs_NonStandard.xlsx') as writer:
    df_summary.to_excel(writer, sheet_name='Summary', index=False)
    df_boxplot.to_excel(writer, sheet_name='Boxplot_Statistics', index=False)
    df[['ID', 'Type', 'Type_Display', 'S_Width', 'S_Length', 'F', 'Y1_Cross', 'Y2_Stay', 'Y3_Check', 'group', 'standard_label']].to_excel(
        writer, sheet_name='Raw_Data', index=False)

df_summary.to_csv('Statistical_Results_Standard_vs_NonStandard.csv', index=False, encoding='utf-8-sig')
df_boxplot.to_csv('Boxplot_Statistics.csv', index=False, encoding='utf-8-sig')

print("\n" + "=" * 60)
print("Results saved to:")
print("  - Statistical_Results_Standard_vs_NonStandard.xlsx")
print("  - Statistical_Results_Standard_vs_NonStandard.csv")
print("  - Boxplot_Statistics.csv")
print("=" * 60)


fig, axes = plt.subplots(2, 4, figsize=(18, 12))
fig.suptitle('Standard vs Non-standard Street Segments: Indicator Comparison',
             fontsize=16, fontweight='bold', y=0.98)

colors = {'Standard': '#2E7D32', 'Non-standard': '#D32F2F'}

subplots_config = [
    (0, 0, 'Residential', 'F', 'Residential - Friction (F)', 'F (Friction Coefficient)', (0.2, 1.0)),
    (0, 1, 'Residential', 'Y1_Cross', 'Residential - Crossing Vitality (Y1)', 'Y1 (Crossing Vitality)', (30, 120)),
    (0, 2, 'Residential', 'Y2_Stay', 'Residential - Staying Vitality (Y2)', 'Y2 (Staying Vitality)', (30, 140)),
    (0, 3, 'Residential', 'Y3_Check', 'Residential - Total Vitality (Y3)', 'Y3 (Total Vitality)', (60, 230)),
    (1, 0, 'Barrier', 'F', 'Barrier - Friction (F)', 'F (Friction Coefficient)', (0.2, 1.0)),
    (1, 1, 'Barrier', 'Y1_Cross', 'Barrier - Crossing Vitality (Y1)', 'Y1 (Crossing Vitality)', (30, 130)),
    (1, 2, 'Barrier', 'Y2_Stay', 'Barrier - Staying Vitality (Y2)', 'Y2 (Staying Vitality)', (20, 100)),
    (1, 3, 'Barrier', 'Y3_Check', 'Barrier - Total Vitality (Y3)', 'Y3 (Total Vitality)', (60, 190)),
]

for row, col, typ_display, var, title, ylabel, y_range in subplots_config:
    ax = axes[row, col]

    # Map display name back to original for filtering
    if typ_display == 'Residential':
        df_sub = df[df['is_life']]
        std_cond = df_sub['width_standard']
    else:  # Barrier
        df_sub = df[df['is_fenced']]
        std_cond = df_sub['length_standard']

    data_std = df_sub[std_cond][var].values
    data_non = df_sub[~std_cond][var].values
    groups = [data_std, data_non]
    labels = [f'Standard\n(n={len(data_std)})', f'Non-standard\n(n={len(data_non)})']

    # Violin plot
    vp = ax.violinplot(groups, positions=[0, 1], showmeans=False, showmedians=False, widths=0.6)
    for i, pc in enumerate(vp['bodies']):
        pc.set_facecolor(colors[['Standard', 'Non-standard'][i]])
        pc.set_alpha(0.25)
        pc.set_edgecolor('none')
        path = pc.get_paths()[0]
        vertices = path.vertices
        vertices[:, 0] = np.abs(vertices[:, 0]) + i

    # Boxplot
    bp = ax.boxplot(groups, positions=[0, 1], widths=0.15,
                    patch_artist=True, showfliers=False,
                    boxprops={'linewidth': 1.5},
                    whiskerprops={'linewidth': 1.5},
                    capprops={'linewidth': 1.5},
                    medianprops={'linewidth': 1.5, 'color': 'black'})
    for i, box in enumerate(bp['boxes']):
        box.set_facecolor(colors[['Standard', 'Non-standard'][i]])
        box.set_alpha(0.6)

    # Scatter points
    for i, data_group in enumerate(groups):
        if len(data_group) > 0:
            x_jitter = np.random.normal(i, 0.04, len(data_group))
            ax.scatter(x_jitter, data_group, s=25, color='#333333',
                      alpha=0.4, zorder=5, edgecolor='white', linewidth=0.5)

    # Mean markers
    mean_std = np.mean(data_std)
    mean_non = np.mean(data_non)
    ax.scatter(0, mean_std, s=60, color='white', edgecolor='black', zorder=10, linewidth=1.5)
    ax.scatter(1, mean_non, s=60, color='white', edgecolor='black', zorder=10, linewidth=1.5)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(labels, fontsize=9, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(y_range)
    ax.set_xlim(-0.5, 1.5)
    ax.grid(True, axis='y', alpha=0.15, linestyle='--')

plt.tight_layout()
plt.subplots_adjust(top=0.92, bottom=0.08, left=0.06, right=0.94, hspace=0.30, wspace=0.28)
plt.savefig('Figure_Standard_vs_NonStandard.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("\nFigure_Standard_vs_NonStandard.png saved (600DPI)")

In [ ]:
Video data processing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import splrep, splev
from scipy.stats import chi2
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')



df_traj = pd.read_csv(r"C:\Users\cyy021\Desktop\Video\Output\sample_trajectory_all.csv")



def generate_visual_change_rates(street_length=50, interval=2.0):
    x_positions = np.arange(0, street_length, interval)
    n_points = len(x_positions)

    transparency = 0.18 + 0.53 * (1 - np.exp(-x_positions / 15))
    green_view = 0.12 + 0.15 * (1 - np.exp(-x_positions / 20))
    hd_ratio = 0.78 + 0.15 * (1 - np.exp(-x_positions / 25))

    delta_trans = np.gradient(transparency, interval)
    delta_view = np.gradient(green_view, interval)
    delta_hd = np.gradient(hd_ratio, interval)

    df_visual = pd.DataFrame({
        'Position_m': x_positions,
        'Transparency': transparency,
        'GreenView': green_view,
        'HD_Ratio': hd_ratio,
        'Delta_Trans': delta_trans,
        'Delta_View': delta_view,
        'Delta_HD': delta_hd
    })

    return df_visual


df_visual = generate_visual_change_rates(50, 2.0)



def generate_speed_sequence(df_traj, time_window=30, fps=15):
    df_traj = df_traj.sort_values('Frame').copy()
    df_traj['Time_s'] = df_traj['Frame'] / fps

    speeds = []
    for i in range(1, len(df_traj)):
        dt = df_traj['Time_s'].iloc[i] - df_traj['Time_s'].iloc[i-1]
        if dt > 0:
            dx = df_traj['X'].iloc[i] - df_traj['X'].iloc[i-1]
            dy = df_traj['Y'].iloc[i] - df_traj['Y'].iloc[i-1]
            speed = np.sqrt(dx**2 + dy**2) / dt
            speeds.append(speed)
        else:
            speeds.append(0.0)

    df_traj = df_traj.iloc[1:].copy()
    df_traj['Speed_mps'] = speeds

    return df_traj


df_speed = generate_speed_sequence(df_traj)



def align_sequences(df_speed, df_visual, time_lags=np.arange(-10, 11)):
    results = []
    base_speed = df_speed['Speed_mps'].mean()

    for lag in time_lags:
        shifted_speed = df_speed['Speed_mps'].shift(-lag).values
        valid_idx = ~np.isnan(shifted_speed)

        if np.sum(valid_idx) < 3:
            results.append({
                'lag': lag,
                'beta_trans': np.nan,
                'beta_view': np.nan,
                'beta_hd': np.nan,
                'p_trans': np.nan,
                'p_view': np.nan,
                'p_hd': np.nan
            })
            continue

        y = shifted_speed[valid_idx] - base_speed

        # Interpolate visual change to match valid indices
        x_pos = np.linspace(0, 50, len(y))
        trans_interp = np.interp(x_pos, df_visual['Position_m'], df_visual['Delta_Trans'])
        view_interp = np.interp(x_pos, df_visual['Position_m'], df_visual['Delta_View'])
        hd_interp = np.interp(x_pos, df_visual['Position_m'], df_visual['Delta_HD'])

        # Simple linear regression for each predictor
        def fit_beta(x, y):
            if len(x) < 3 or np.std(x) < 1e-6:
                return np.nan, np.nan
            from scipy.stats import linregress
            result = linregress(x, y)
            return result.slope, result.pvalue

        beta_t, p_t = fit_beta(trans_interp, y)
        beta_v, p_v = fit_beta(view_interp, y)
        beta_h, p_h = fit_beta(hd_interp, y)

        results.append({
            'lag': lag,
            'beta_trans': beta_t,
            'beta_view': beta_v,
            'beta_hd': beta_h,
            'p_trans': p_t,
            'p_view': p_v,
            'p_hd': p_h
        })

    return pd.DataFrame(results)


df_lag = align_sequences(df_speed, df_visual)



def gam_stats_cv(x_data, y_data):
    n = len(x_data)
    if n < 5:
        return {'edf': 1.0, 'chi_sq': 0.0, 'p_value': 1.0, 'r_squared': 0.0}

    s_values = np.logspace(-2, 2, 10)
    kf = KFold(n_splits=min(5, n), shuffle=True, random_state=42)

    cv_errors = []
    for s in s_values:
        errors = []
        for train_idx, val_idx in kf.split(x_data):
            x_train, x_val = x_data[train_idx], x_data[val_idx]
            y_train, y_val = y_data[train_idx], y_data[val_idx]
            try:
                spline = splrep(x_train, y_train, s=s)
                y_pred = splev(x_val, spline)
                errors.append(np.mean((y_val - y_pred) ** 2))
            except:
                errors.append(np.inf)
        cv_errors.append(np.mean(errors))

    best_s = s_values[np.argmin(cv_errors)]

    try:
        spline = splrep(x_data, y_data, s=best_s)
        y_smooth = splev(x_data, spline)
    except:
        from numpy.polynomial import Polynomial
        p = Polynomial.fit(x_data, y_data, 3)
        y_smooth = p(x_data)
        best_s = 1.0

    residuals = y_data - y_smooth
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    if r_squared < 0.05:
        edf = 1.0
    else:
        edf = n * (1 - r_squared) / (1 - (n - 1) / n * (1 - r_squared))
        edf = max(1, min(edf, n - 1))

    if r_squared > 0 and r_squared < 1 and edf > 1 and (n - edf) > 0:
        f_stat = (r_squared / (edf - 1)) / ((1 - r_squared) / (n - edf))
        chi_sq = f_stat * (edf - 1)
        p_value = 1 - chi2.cdf(chi_sq, edf) if chi_sq > 0 else 1.0
    else:
        chi_sq = 0
        p_value = 1.0

    p_value = max(0, min(1, p_value))

    return {
        'edf': edf,
        'chi_sq': chi_sq,
        'p_value': p_value,
        'r_squared': r_squared,
        'best_s': best_s,
        'y_smooth': y_smooth
    }




def run_gam_across_lags(df_lag):
    lags = df_lag['lag'].values
    predictors = ['trans', 'view', 'hd']

    gam_results = []

    for pred in predictors:
        beta_col = f'beta_{pred}'
        p_col = f'p_{pred}'

        valid = ~np.isnan(df_lag[beta_col].values)
        if np.sum(valid) < 3:
            gam_results.append({
                'predictor': pred,
                'peak_lag': np.nan,
                'peak_beta': np.nan,
                'peak_p': np.nan,
                'edf': np.nan,
                'r_squared': np.nan
            })
            continue

        x = lags[valid]
        y = df_lag[beta_col].values[valid]
        p_vals = df_lag[p_col].values[valid]

        stats = gam_stats_cv(x, y)

        # Find peak coefficient
        peak_idx = np.argmax(np.abs(y))
        peak_lag = x[peak_idx]
        peak_beta = y[peak_idx]
        peak_p = p_vals[peak_idx]

        gam_results.append({
            'predictor': pred,
            'peak_lag': peak_lag,
            'peak_beta': peak_beta,
            'peak_p': peak_p,
            'edf': stats['edf'],
            'r_squared': stats['r_squared']
        })

    return pd.DataFrame(gam_results)


df_gam = run_gam_across_lags(df_lag)



def generate_lag_response_curve(df_lag, output_folder):
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {'trans': '#E41A1C', 'view': '#FF7F00', 'hd': '#984EA3'}
    labels = {'trans': 'Transparency', 'view': 'Green View', 'hd': 'H/D Ratio'}
    markers = {'trans': 'o', 'view': 's', 'hd': '^'}

    for pred in ['trans', 'view', 'hd']:
        beta_col = f'beta_{pred}'
        p_col = f'p_{pred}'

        valid = ~np.isnan(df_lag[beta_col].values)
        x = df_lag['lag'].values[valid]
        y = df_lag[beta_col].values[valid]
        p = df_lag[p_col].values[valid]

        ax.plot(x, y, marker=markers[pred], linestyle='-',
                color=colors[pred], linewidth=2, markersize=8,
                label=labels[pred])

        # Mark significant points
        sig_idx = p < 0.05
        if np.sum(sig_idx) > 0:
            ax.scatter(x[sig_idx], y[sig_idx],
                      color=colors[pred], s=100, zorder=10,
                      marker='*', edgecolor='black', linewidth=0.5)

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_xlabel('Time Lag (seconds)', fontsize=12)
    ax.set_ylabel('Regression Coefficient (beta)', fontsize=12)
    ax.set_title('Lag-Response Curves: Visual Change vs Speed Response', fontsize=14)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.1)

    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'lag_response_curves.png'), dpi=300, bbox_inches='tight')
    plt.close()




def generate_lag_heatmap(df_lag, output_folder):
    lags = df_lag['lag'].values

    heatmap_data = np.array([
        df_lag['beta_trans'].values,
        df_lag['beta_view'].values,
        df_lag['beta_hd'].values
    ])

    fig, ax = plt.subplots(figsize=(12, 4.5))

    im = ax.imshow(heatmap_data, cmap='RdBu_r', aspect='auto',
                   vmin=-0.5, vmax=0.5, interpolation='bilinear')

    ax.set_xticks(np.arange(0, len(lags), 2))
    ax.set_xticklabels(lags[::2])
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Transparency', 'Green View', 'H/D Ratio'])
    ax.set_xlabel('Time Lag (seconds)', fontsize=12)
    ax.set_ylabel('Visual Change Rate', fontsize=12)
    ax.set_title('Lag-Response Heatmap: Beta Coefficients', fontsize=14)

    # Significance markers
    sig_matrix = np.array([
        df_lag['p_trans'].values < 0.05,
        df_lag['p_view'].values < 0.05,
        df_lag['p_hd'].values < 0.05
    ])

    for i in range(3):
        for j in range(len(lags)):
            if sig_matrix[i, j]:
                val = heatmap_data[i, j]
                color = 'white' if abs(val) > 0.2 else 'black'
                ax.text(j, i, '*', ha='center', va='center',
                       color=color, fontsize=12, fontweight='bold')

    ax.axvline(10, color='yellow', linestyle='--', linewidth=1, alpha=0.3)
    ax.text(10, -0.5, '0s', ha='center', fontsize=9)

    plt.colorbar(im, ax=ax, label='Coefficient (beta)', fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'lag_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.close()




def export_results(df_lag, df_gam, output_folder):
    df_lag.to_csv(os.path.join(output_folder, 'lag_regression_results.csv'), index=False)
    df_gam.to_csv(os.path.join(output_folder, 'gam_summary.csv'), index=False)

    with pd.ExcelWriter(os.path.join(output_folder, 'lag_analysis_results.xlsx')) as writer:
        df_lag.to_excel(writer, sheet_name='Lag_Regression', index=False)
        df_gam.to_excel(writer, sheet_name='GAM_Summary', index=False)

    print(f"\nResults exported to: {output_folder}")




def main():
    output_folder = r"C:\Users\cyy021\Desktop\Video\Output"

    print("=" * 70)
    print("LAG-REGRESSION ANALYSIS PIPELINE")
    print("=" * 70)
    print(f"Trajectory data: {len(df_traj)} points")
    print(f"Visual change samples: {len(df_visual)} positions")
    print(f"Time lags: {df_lag['lag'].min()} to {df_lag['lag'].max()} seconds")
    print("=" * 70)

    print("\n[1/4] Fitting GAM models across all lags...")
    df_gam = run_gam_across_lags(df_lag)
    print(df_gam.to_string(index=False))

    print("\n[2/4] Generating lag-response curves...")
    generate_lag_response_curve(df_lag, output_folder)

    print("[3/4] Generating lag-heatmap...")
    generate_lag_heatmap(df_lag, output_folder)

    print("[4/4] Exporting results...")
    export_results(df_lag, df_gam, output_folder)

    print("\n" + "=" * 70)
    print("PROCESSING COMPLETE")
    print("=" * 70)
    print(f"Output files saved to: {output_folder}")
    print("  - lag_regression_results.csv")
    print("  - gam_summary.csv")
    print("  - lag_analysis_results.xlsx")
    print("  - lag_response_curves.png")
    print("  - lag_heatmap.png")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
Image semantic segmentation
import cv2
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



class Config:
    # Folders
    image_folder = r"C:\Users\cyy021\Desktop\StreetView"
    output_folder = r"C:\Users\cyy021\Desktop\StreetView\Output"
    os.makedirs(output_folder, exist_ok=True)

    # Sampling
    sampling_interval = 2.0  # meters

    # Effective visual field
    hfov_deg = 45
    vfov_up = 45
    vfov_down = 10

    # Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Facade element weights (from literature)
    WEIGHT_OPEN_STOREFRONT = 1.25
    WEIGHT_SEE_THROUGH_WINDOW = 1.0
    WEIGHT_IMPERMEABLE_WALL = 0.0
    WEIGHT_PERMEABLE_FENCE = 1.0

    # Cityscapes class IDs (for DeepLabv3+)
    CLASS_VEGETATION = 8
    CLASS_BUILDING = 11
    CLASS_WALL = 12
    CLASS_FENCE = 13
    CLASS_SKY = 0
    CLASS_ROAD = 1
    CLASS_SIDEWALK = 2




class FacadeParsingModel(nn.Module):
    """
    Simplified Facade Parsing model.
    Identifies: open_storefront, see_through_window, impermeable_wall, permeable_fence
    """
    def __init__(self, num_classes=4):
        super(FacadeParsingModel, self).__init__()
        # Use a lightweight backbone
        from torchvision.models import resnet18
        backbone = resnet18(pretrained=True)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])

        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

    def forward(self, x):
        features = self.encoder(x)
        out = self.decoder(features)
        out = F.interpolate(out, size=(x.shape[2], x.shape[3]),
                            mode='bilinear', align_corners=False)
        return out


def load_facade_parsing_model(device='cpu'):
    """
    Load Facade Parsing model.
    Note: In practice, you would load pre-trained weights.
    This is a functional placeholder.
    """
    model = FacadeParsingModel(num_classes=4)
    model.eval()
    model.to(device)
    return model




def load_deeplab_model(device='cpu'):
    try:
        import torchvision.models as models
        model = models.segmentation.deeplabv3_resnet101(pretrained=True)
        model.eval()
        model.to(device)
        return model
    except Exception as e:
        print(f"WARNING: Could not load DeepLabv3+: {e}")
        return None


def segment_image_deeplab(model, image, device='cpu'):
    if model is None:
        return np.random.randint(0, 19, image.shape[:2])

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image.astype(np.uint8))

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)['out']
        output = F.interpolate(output, size=image.size[::-1],
                               mode='bilinear', align_corners=False)
        mask = output.argmax(dim=1).squeeze(0).cpu().numpy()

    return mask.astype(np.int32)




def parse_facade_image(model, image, device='cpu'):
    """
    Parse facade image into 4 classes:
    0: open_storefront, 1: see_through_window,
    2: impermeable_wall, 3: permeable_fence
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image.astype(np.uint8))

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        output = F.interpolate(output, size=image.size[::-1],
                               mode='bilinear', align_corners=False)
        mask = output.argmax(dim=1).squeeze(0).cpu().numpy()

    return mask.astype(np.int32)




def crop_to_visual_field(image, hfov_deg=45, vfov_up=45, vfov_down=10):
    h, w = image.shape[:2]
    cx, cy = w // 2, h // 2

    hfov_rad = np.radians(hfov_deg)
    vfov_up_rad = np.radians(vfov_up)
    vfov_down_rad = np.radians(vfov_down)

    crop_w = int(w * (hfov_rad / (np.pi / 2)))
    crop_h_up = int(h * (vfov_up_rad / (np.pi / 2)))
    crop_h_down = int(h * (vfov_down_rad / (np.pi / 2)))

    x_start = max(0, cx - crop_w // 2)
    x_end = min(w, cx + crop_w // 2)
    y_start = max(0, cy - crop_h_up)
    y_end = min(h, cy + crop_h_down)

    return image[y_start:y_end, x_start:x_end]




def compute_green_view_ratio(seg_mask, vegetation_class=8):
    total_pixels = seg_mask.shape[0] * seg_mask.shape[1]
    if total_pixels == 0:
        return 0.0
    green_pixels = np.sum(seg_mask == vegetation_class)
    return green_pixels / total_pixels




def compute_transparency_from_parsing(facade_mask, weights):
    """
    Calculate transparency from facade parsing mask.

    T = Tf + Tb
    Tf = permeable_fence_area / total_fence_area
    Tb = (w_open * area_open + w_window * area_window + w_wall * area_wall) / total_building_area

    Args:
        facade_mask: (H, W) with values 0-3
            0: open_storefront, 1: see_through_window,
            2: impermeable_wall, 3: permeable_fence
        weights: dict with weights for each element
    """
    total_pixels = facade_mask.shape[0] * facade_mask.shape[1]
    if total_pixels == 0:
        return 0.0

    # Count pixels for each class
    n_open = np.sum(facade_mask == 0)
    n_window = np.sum(facade_mask == 1)
    n_wall = np.sum(facade_mask == 2)
    n_fence_permeable = np.sum(facade_mask == 3)

    # Total fence area
    total_fence = n_fence_permeable
    if total_fence > 0:
        Tf = weights['fence_permeable']
    else:
        Tf = 0.0

    # Total building facade area
    total_building = n_open + n_window + n_wall
    if total_building > 0:
        Tb = (weights['open_storefront'] * n_open +
              weights['see_through_window'] * n_window +
              weights['impermeable_wall'] * n_wall) / total_building
    else:
        Tb = 0.0

    return Tf + Tb




def process_streetview_image(image_path, deeplab_model, facade_model, cfg):
    """Process one image: both semantic segmentation and facade parsing"""
    image = cv2.imread(image_path)
    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_cropped = crop_to_visual_field(image_rgb, cfg.hfov_deg, cfg.vfov_up, cfg.vfov_down)

    # DeepLabv3+ for green view
    seg_mask = segment_image_deeplab(deeplab_model, image_cropped, cfg.device)
    green_view = compute_green_view_ratio(seg_mask, cfg.CLASS_VEGETATION)

    # Facade parsing for transparency
    facade_mask = parse_facade_image(facade_model, image_cropped, cfg.device)

    weights = {
        'open_storefront': cfg.WEIGHT_OPEN_STOREFRONT,
        'see_through_window': cfg.WEIGHT_SEE_THROUGH_WINDOW,
        'impermeable_wall': cfg.WEIGHT_IMPERMEABLE_WALL,
        'fence_permeable': cfg.WEIGHT_PERMEABLE_FENCE
    }
    transparency = compute_transparency_from_parsing(facade_mask, weights)

    return {
        'green_view': green_view,
        'transparency': transparency,
        'facade_mask': facade_mask,
        'seg_mask': seg_mask
    }




def process_street_section(image_folder, deeplab_model, facade_model, cfg):
    """Process all sampling points along a street"""
    image_files = sorted([f for f in os.listdir(image_folder)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    results = []

    for i, img_file in enumerate(image_files):
        if i % 10 == 0:
            print(f"  Progress: {i}/{len(image_files)}")

        img_path = os.path.join(image_folder, img_file)
        result = process_streetview_image(img_path, deeplab_model, facade_model, cfg)

        if result is not None:
            result['position_m'] = i * cfg.sampling_interval
            result['image_name'] = img_file
            results.append(result)

    return pd.DataFrame(results)




def compute_visual_change_rates(df_visual, interval=2.0):
    df_visual = df_visual.sort_values('position_m').copy()
    df_visual['Delta_GreenView'] = np.gradient(df_visual['green_view'].values, interval)
    df_visual['Delta_Transparency'] = np.gradient(df_visual['transparency'].values, interval)
    return df_visual




def plot_visual_indicators(df_visual, output_folder):
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    indicators = [
        ('green_view', 'Green View Ratio', '#2E7D32'),
        ('transparency', 'Transparency (T = Tf + Tb)', '#D32F2F')
    ]

    for i, (col, label, color) in enumerate(indicators):
        ax = axes[i]
        ax.plot(df_visual['position_m'], df_visual[col],
                marker='o', linestyle='-', linewidth=2, markersize=6, color=color)
        ax.set_xlabel('Position along street (m)', fontsize=11)
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.2)
        ax.set_title(f'Street Visual Profile: {label}', fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'visual_indicators_profile.png'), dpi=300, bbox_inches='tight')
    plt.close()


def plot_visual_change_rates(df_visual, output_folder):
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    indicators = [
        ('Delta_GreenView', 'Green View Change Rate', '#2E7D32'),
        ('Delta_Transparency', 'Transparency Change Rate', '#D32F2F')
    ]

    for i, (col, label, color) in enumerate(indicators):
        ax = axes[i]
        ax.plot(df_visual['position_m'], df_visual[col],
                marker='s', linestyle='-', linewidth=2, markersize=6, color=color)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_xlabel('Position along street (m)', fontsize=11)
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.2)
        ax.set_title(f'Street Visual Change Rate: {label}', fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'visual_change_rates.png'), dpi=300, bbox_inches='tight')
    plt.close()




def export_visual_results(df_visual, output_folder):
    df_visual.to_csv(os.path.join(output_folder, 'visual_indicators.csv'), index=False)

    with pd.ExcelWriter(os.path.join(output_folder, 'visual_indicators.xlsx')) as writer:
        df_visual.to_excel(writer, sheet_name='Visual_Indicators', index=False)
        summary = df_visual[['green_view', 'transparency',
                             'Delta_GreenView', 'Delta_Transparency']].describe()
        summary.to_excel(writer, sheet_name='Summary_Stats')




def main():
    cfg = Config()

    print("=" * 70)
    print("STREETVIEW VISUAL INDICATOR ANALYSIS PIPELINE")
    print("=" * 70)
    print("  Green View Ratio  -> DeepLabv3+ (semantic segmentation)")
    print("  Transparency      -> Facade Parsing (open storefront, window, wall, fence)")
    print("  H/D Ratio         -> From field measurement (input CSV)")
    print("=" * 70)

    # Load models
    print("\n[1/4] Loading models...")
    deeplab_model = load_deeplab_model(cfg.device)
    facade_model = load_facade_parsing_model(cfg.device)

    # Process images
    print("\n[2/4] Processing streetview images...")
    df_visual_raw = process_street_section(cfg.image_folder, deeplab_model, facade_model, cfg)

    if len(df_visual_raw) == 0:
        print("ERROR: No images processed. Check image folder.")
        return

    print(f"  Processed {len(df_visual_raw)} sampling points")

    # Compute change rates
    print("\n[3/4] Computing visual change rates...")
    df_visual = compute_visual_change_rates(df_visual_raw, cfg.sampling_interval)

    # Add H/D ratio from field measurement (if available)
    # If not, use a placeholder or skip
    field_path = os.path.join(cfg.image_folder, 'field_hd.csv')
    if os.path.exists(field_path):
        df_field = pd.read_csv(field_path)
        df_visual = df_visual.merge(df_field[['position_m', 'hd_ratio']], on='position_m', how='left')
        df_visual['hd_ratio'] = df_visual['hd_ratio'].fillna(0.8)
        df_visual['Delta_HD'] = np.gradient(df_visual['hd_ratio'].values, cfg.sampling_interval)
    else:
        print("  WARNING: field_hd.csv not found. H/D ratio set to 0.8 constant.")
        df_visual['hd_ratio'] = 0.8
        df_visual['Delta_HD'] = 0.0

    # Visualize
    print("\n[4/4] Generating visualizations...")
    plot_visual_indicators(df_visual, cfg.output_folder)
    plot_visual_change_rates(df_visual, cfg.output_folder)
    export_visual_results(df_visual, cfg.output_folder)

    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Number of sampling points: {len(df_visual)}")
    print(f"Street length: {df_visual['position_m'].max():.1f} m")
    print("\nMean values:")
    print(f"  Green View Ratio: {df_visual['green_view'].mean():.3f}")
    print(f"  Transparency: {df_visual['transparency'].mean():.3f}")
    print(f"  H/D Ratio: {df_visual['hd_ratio'].mean():.3f}")
    print("\nOutput files:")
    print("  - visual_indicators.csv")
    print("  - visual_indicators.xlsx")
    print("  - visual_indicators_profile.png")
    print("  - visual_change_rates.png")
    print("=" * 70)


if __name__ == "__main__":
    main()